# FairLend: Credit Default Prediction under Fairness Constraints
**Google Colab notebook** (BTECH CAP 501, Project Life Cycle Management)

Run **Runtime > Run all**. It takes a few minutes on the free CPU runtime and reproduces the numbers in the report:
1. **Part 1**: synthetic dataset, 60/20/20 split, baseline, XGBoost, LightGBM, Optuna (40 trials), MLflow, calibration, fairness mitigation
2. **Part 2**: EDA, data quality, Fairlearn, group calibration, error analysis, SHAP, DiCE, drift (PSI), model packaging
3. **Part 3**: live scoring, MLflow run table and a download of all results

> The dataset is **synthetic** with deliberately built-in gender and region bias. Seed 42 makes results reproducible.

## 0. Setup

In [ ]:
!pip -q install optuna mlflow xgboost lightgbm fairlearn shap dice-ml scipy

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
os.makedirs('results', exist_ok=True); os.makedirs('deploy', exist_ok=True)
%matplotlib inline

## Part 1. Data, models, Optuna, MLflow and fairness

In [ ]:
"""FairLend: Credit default prediction under fairness constraints (synthetic MFI data).
Runs end-to-end: data -> split -> baseline -> XGB/LGBM -> Optuna -> MLflow -> calibration -> fairness.
All numbers in the presentation come from this script's output (results/*.json, results/*.png)."""
import json, time, os, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import optuna, mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score, accuracy_score,
                             confusion_matrix, brier_score_loss, roc_curve)
import xgboost as xgb, lightgbm as lgb
warnings.filterwarnings("ignore"); optuna.logging.set_verbosity(optuna.logging.WARNING)

OUT = "results"; os.makedirs(OUT, exist_ok=True)
SEED, N, N_TRIALS = 42, 50000, 40
rng = np.random.default_rng(SEED)
res = {}



### 1. Synthetic MFI dataset (same generative story as the FairLend web app, with built-in bias)

In [ ]:
f = (rng.random(N) < .45).astype(int)          # 1 = female
r = (rng.random(N) < .55).astype(int)          # 1 = rural
inc = np.exp(rng.normal(0, .4, N) + 2.6 - .25*r - .2*f)
rc = np.clip(.65 - .12*r - .05*f + .18*rng.normal(size=N), 0, 1)
mm = np.maximum(0, np.round(np.exp(rng.normal(0, .5, N) + 2.2 - .3*r - .15*f)))
ut = np.clip(.75 - .08*r + .15*rng.normal(size=N), 0, 1)
shg = np.clip(.7 + .1*f + .15*rng.normal(size=N), 0, 1)
prev = (rng.random(N) < .12 + .05*r).astype(int)
loan = np.clip(np.exp(rng.normal(0, .5, N) + 2.7), 3, 60)
inf = (rng.random(N) < .45 + .2*r + .15*f).astype(int)
dev = (rng.random(N) < .35 + .3*r + .1*f).astype(int)
z = -2.3 + 1.5*(-2.4*(rc-.6) - 2*(ut-.7) - 1.6*(shg-.7) + prev + .035*loan - .05*inc + .35*inf + .3*r + .2*f) + .3*rng.normal(size=N)
y = (rng.random(N) < 1/(1+np.exp(-z))).astype(int)
df = pd.DataFrame(dict(recharge_regularity=rc, mobile_money_txn_log=np.log1p(mm), utility_on_time=ut,
                       shg_repayment=shg, earlier_unpaid_loan=prev, loan_amount_k=loan, monthly_income_k=inc,
                       informal_income=inf, basic_phone=dev, default=y))
prot = pd.DataFrame(dict(female=f, rural=r))      # protected attributes: audit only, never model inputs
FEATS = [c for c in df.columns if c != "default"]
df.assign(**prot).to_csv(f"{OUT}/dataset.csv", index=False)
res["data"] = dict(n=N, n_features=len(FEATS), features=FEATS, default_rate=float(y.mean()),
                   default_rate_by=dict(male=float(y[f==0].mean()), female=float(y[f==1].mean()),
                                        urban=float(y[r==0].mean()), rural=float(y[r==1].mean())),
                   protected_share=dict(female=float(f.mean()), rural=float(r.mean())))

### 2. Split 60/20/20 stratified on target x group (test untouched until final)

In [ ]:
strat = df["default"]*4 + prot.female*2 + prot.rural
idx = np.arange(N)
i_tr, i_tmp = train_test_split(idx, test_size=.4, stratify=strat, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=.5, stratify=strat.iloc[i_tmp], random_state=SEED)
X, Y = df[FEATS].values, df["default"].values
sc = StandardScaler().fit(X[i_tr])                 # scaler fitted on TRAIN only (no leakage)
Xs = sc.transform(X)
res["split"] = dict(train=len(i_tr), val=len(i_va), test=len(i_te),
                    default_rate=[float(Y[i].mean()) for i in (i_tr, i_va, i_te)])

### helpers

In [ ]:
def ks_stat(y_, p_):
    o = np.argsort(p_); yy = y_[o]; P, Nn = yy.sum(), len(yy)-yy.sum()
    return float(np.max(np.abs(np.cumsum(yy)/P - np.cumsum(1-yy)/Nn))*100)

def metrics(y_, p_, thr):
    pred = (p_ >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(y_, pred).ravel()
    return dict(auc=float(roc_auc_score(y_, p_)), ks=ks_stat(y_, p_), accuracy=float(accuracy_score(y_, pred)),
                precision=float(precision_score(y_, pred)), recall=float(recall_score(y_, pred)),
                f1=float(f1_score(y_, pred)), brier=float(brier_score_loss(y_, p_)),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), thr=float(thr))

def best_f1_thr(y_, p_):
    ts = np.linspace(.05, .8, 76); return float(ts[np.argmax([f1_score(y_, p_ >= t) for t in ts])])

def reweigh(idx_):
    g = (prot.female.values*2 + prot.rural.values)[idx_]; yy = Y[idx_]; n = len(idx_)
    pg = np.bincount(g, minlength=4)/n; py = np.bincount(yy, minlength=2)/n
    pgy = np.zeros((4, 2))
    for a in range(4):
        for b in range(2): pgy[a, b] = ((g == a) & (yy == b)).sum()/n
    return np.array([pg[a]*py[b]/pgy[a, b] for a, b in zip(g, yy)])

def fairness(p_, idx_, thr_by_group=None, thr=None):
    """approve = p < threshold. DIR=min/max approval-rate ratio, DPD=max gap, EOD=max gap in approval of non-defaulters."""
    g = (prot.female.values*2 + prot.rural.values)[idx_]; yy = Y[idx_]
    t = np.array([thr_by_group[a] for a in g]) if thr_by_group is not None else np.full(len(idx_), thr)
    ap = p_ < t; out = dict(dir=1.0, dpd=0.0, eod=0.0)
    for name, grp in (("gender", prot.female.values[idx_]), ("region", prot.rural.values[idx_])):
        a, b = grp == 1, grp == 0; ra, rb = ap[a].mean(), ap[b].mean()
        out["dir"] = min(out["dir"], min(ra, rb)/max(ra, rb)); out["dpd"] = max(out["dpd"], abs(ra-rb))
        tpr = lambda m: ap[m & (yy == 0)].mean(); out["eod"] = max(out["eod"], abs(tpr(a)-tpr(b)))
        out[f"rate_{name}"] = dict(group1=float(ra), group0=float(rb))
    out["approval_overall"] = float(ap.mean())
    out["group_rates"] = [float(ap[g == k].mean()) for k in range(4)]
    return {k: (float(v) if not isinstance(v, (dict, list)) else v) for k, v in out.items()}

### 3. MLflow setup

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath('mlflow.db')}")
mlflow.set_experiment("FairLend-credit-default")
def log_run(name, model_type, params, val_m, extra_tags=None, test_m=None, dur=None):
    with mlflow.start_run(run_name=name):
        mlflow.log_params({k: (round(v, 6) if isinstance(v, float) else v) for k, v in params.items()})
        mlflow.set_tags({"model": model_type, "dataset": "synthetic_mfi_v1_n50000", "split": "60/20/20", "seed": SEED, **(extra_tags or {})})
        for k in ("auc", "ks", "f1", "precision", "recall", "accuracy", "brier"): mlflow.log_metric(f"val_{k}", val_m[k])
        if test_m:
            for k in ("auc", "ks", "f1", "precision", "recall", "accuracy", "brier"): mlflow.log_metric(f"test_{k}", test_m[k])
        if dur is not None: mlflow.log_metric("train_seconds", dur)

### 4. Baseline + default models (validation only)

In [ ]:
t0 = time.time()
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xs[i_tr], Y[i_tr]); t_lr = time.time()-t0
p_lr_va = lr.predict_proba(Xs[i_va])[:, 1]; thr_lr = best_f1_thr(Y[i_va], p_lr_va)
m_lr_va = metrics(Y[i_va], p_lr_va, thr_lr)
log_run("baseline_logreg", "LogisticRegression", dict(C=1.0, max_iter=1000, features=len(FEATS)), m_lr_va, dur=t_lr)

t0 = time.time()
xg0 = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=.1, random_state=SEED, n_jobs=4, verbosity=0).fit(X[i_tr], Y[i_tr]); t_x0 = time.time()-t0
p_x0_va = xg0.predict_proba(X[i_va])[:, 1]; m_x0_va = metrics(Y[i_va], p_x0_va, best_f1_thr(Y[i_va], p_x0_va))
log_run("xgb_default", "XGBoost", dict(n_estimators=300, max_depth=6, learning_rate=.1), m_x0_va, dur=t_x0)

t0 = time.time()
lg0 = lgb.LGBMClassifier(n_estimators=300, num_leaves=31, learning_rate=.1, random_state=SEED, n_jobs=4, verbose=-1).fit(X[i_tr], Y[i_tr]); t_l0 = time.time()-t0
p_l0_va = lg0.predict_proba(X[i_va])[:, 1]; m_l0_va = metrics(Y[i_va], p_l0_va, best_f1_thr(Y[i_va], p_l0_va))
log_run("lgbm_default", "LightGBM", dict(n_estimators=300, num_leaves=31, learning_rate=.1), m_l0_va, dur=t_l0)
res["val_defaults"] = dict(logreg=m_lr_va, xgb=m_x0_va, lgbm=m_l0_va, train_seconds=dict(logreg=t_lr, xgb=t_x0, lgbm=t_l0))

### 5. Optuna (objective = validation AUC; test set never touched)

In [ ]:
SPACE = dict(n_estimators=(100, 600), max_depth=(2, 8), learning_rate=(.01, .3), subsample=(.5, 1.0),
             colsample_bytree=(.5, 1.0), reg_lambda=(1e-3, 10.0), min_child_weight=(1, 20))
def objective(trial):
    p = dict(n_estimators=trial.suggest_int("n_estimators", *SPACE["n_estimators"]),
             max_depth=trial.suggest_int("max_depth", *SPACE["max_depth"]),
             learning_rate=trial.suggest_float("learning_rate", *SPACE["learning_rate"], log=True),
             subsample=trial.suggest_float("subsample", *SPACE["subsample"]),
             colsample_bytree=trial.suggest_float("colsample_bytree", *SPACE["colsample_bytree"]),
             reg_lambda=trial.suggest_float("reg_lambda", *SPACE["reg_lambda"], log=True),
             min_child_weight=trial.suggest_int("min_child_weight", *SPACE["min_child_weight"]))
    t0 = time.time(); m = xgb.XGBClassifier(**p, random_state=SEED, n_jobs=4, verbosity=0).fit(X[i_tr], Y[i_tr]); d = time.time()-t0
    pv = m.predict_proba(X[i_va])[:, 1]; vm = metrics(Y[i_va], pv, best_f1_thr(Y[i_va], pv))
    log_run(f"optuna_trial_{trial.number:02d}", "XGBoost", p, vm, {"stage": "optuna"}, dur=d)
    return vm["auc"]
t0 = time.time()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS); t_opt = time.time()-t0
bp = study.best_params
res["optuna"] = dict(n_trials=N_TRIALS, best_value=study.best_value, best_trial=study.best_trial.number, best_params=bp,
                     total_seconds=t_opt, space={k: list(v) for k, v in SPACE.items()},
                     values=[t.value for t in study.trials])
trials_df = study.trials_dataframe(); trials_df.to_csv(f"{OUT}/optuna_trials.csv", index=False)
imp = optuna.importance.get_param_importances(study); res["optuna"]["importance"] = imp

t0 = time.time()
xgb_best = xgb.XGBClassifier(**bp, random_state=SEED, n_jobs=4, verbosity=0).fit(X[i_tr], Y[i_tr]); t_xb = time.time()-t0
p_xb_va = xgb_best.predict_proba(X[i_va])[:, 1]; thr_xb = best_f1_thr(Y[i_va], p_xb_va)
m_xb_va = metrics(Y[i_va], p_xb_va, thr_xb)

### 6. Calibration (isotonic fitted on VALIDATION)

In [ ]:
iso = IsotonicRegression(out_of_bounds="clip").fit(p_xb_va, Y[i_va])

### 7. FINAL TEST evaluation (single pass)

In [ ]:
P = {}
P["logreg"] = lr.predict_proba(Xs[i_te])[:, 1]
P["xgb_default"] = xg0.predict_proba(X[i_te])[:, 1]
P["lgbm_default"] = lg0.predict_proba(X[i_te])[:, 1]
P["xgb_tuned"] = xgb_best.predict_proba(X[i_te])[:, 1]
P["xgb_tuned_cal"] = iso.predict(P["xgb_tuned"])
thr_val = dict(logreg=thr_lr, xgb_default=best_f1_thr(Y[i_va], p_x0_va), lgbm_default=best_f1_thr(Y[i_va], p_l0_va),
               xgb_tuned=thr_xb, xgb_tuned_cal=best_f1_thr(Y[i_va], iso.predict(p_xb_va)))
res["test"] = {k: metrics(Y[i_te], P[k], thr_val[k]) for k in P}
res["test_train_seconds"] = dict(logreg=t_lr, xgb_default=t_x0, lgbm_default=t_l0, xgb_tuned=t_xb, optuna_total=t_opt)
res["val_tuned"] = m_xb_va
# train-vs-val AUC gap (overfitting check)
res["overfit"] = dict(logreg=[float(roc_auc_score(Y[i_tr], lr.predict_proba(Xs[i_tr])[:, 1])), m_lr_va["auc"]],
                      xgb_default=[float(roc_auc_score(Y[i_tr], xg0.predict_proba(X[i_tr])[:, 1])), m_x0_va["auc"]],
                      xgb_tuned=[float(roc_auc_score(Y[i_tr], xgb_best.predict_proba(X[i_tr])[:, 1])), m_xb_va["auc"]])
log_run("final_logreg_TEST", "LogisticRegression", dict(C=1.0), m_lr_va, {"stage": "final"}, test_m=res["test"]["logreg"], dur=t_lr)
log_run("final_xgb_tuned_TEST", "XGBoost", bp, m_xb_va, {"stage": "final"}, test_m=res["test"]["xgb_tuned"], dur=t_xb)
log_run("final_xgb_tuned_isotonic_TEST", "XGBoost+Isotonic", bp, m_xb_va, {"stage": "final"}, test_m=res["test"]["xgb_tuned_cal"], dur=t_xb)

### 8. Fairness: reweighing + group-specific thresholds

In [ ]:
# Model chosen on VALIDATION AUC (not test). Fairness is applied to that model; tuned XGB shown as secondary.
t0 = time.time()
SEL = "logreg" if m_lr_va["auc"] >= m_xb_va["auc"] else "xgb_tuned"
res["selected_model"] = SEL
w_tr = reweigh(i_tr)
lr_rw = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xs[i_tr], Y[i_tr], sample_weight=w_tr)
xgb_rw = xgb.XGBClassifier(**bp, random_state=SEED, n_jobs=4, verbosity=0).fit(X[i_tr], Y[i_tr], sample_weight=w_tr)
P["logreg_rw"] = lr_rw.predict_proba(Xs[i_te])[:, 1]; pv_lr_rw = lr_rw.predict_proba(Xs[i_va])[:, 1]
P["xgb_tuned_rw"] = xgb_rw.predict_proba(X[i_te])[:, 1]; pv_rw = xgb_rw.predict_proba(X[i_va])[:, 1]
APPROVE = .70
G_te = (prot.female.values*2 + prot.rural.values)[i_te]; G_va = (prot.female.values*2 + prot.rural.values)[i_va]
def group_thr(p_va, lam):
    t0_ = np.quantile(p_va, APPROVE); out = []
    for g in range(4):
        pg = p_va[G_va == g]; nat = (pg < t0_).mean()
        out.append(np.quantile(pg, min(.999, (1-lam)*nat + lam*APPROVE)))
    return out
def run_grid(pv_no, pt_no, pv_rw_, pt_rw_):
    grid = []
    for rw, (pv_, pt_) in enumerate([(pv_no, pt_no), (pv_rw_, pt_rw_)]):
        for lam in np.linspace(0, 1, 11):
            th = group_thr(pv_, lam); fr = fairness(pt_, i_te, thr_by_group=th)
            yp = Y[i_te]; ap = pt_ < np.array([th[g] for g in G_te]); pred = (~ap).astype(int)
            tn, fp, fn, tp = confusion_matrix(yp, pred).ravel(); ba = (tn/(tn+fp) + tp/(tp+fn))/2
            prec = tp/(tp+fp); rec = tp/(tp+fn)
            grid.append(dict(reweigh=rw, lam=float(lam), ba=float(ba), f1=float(2*prec*rec/(prec+rec)), auc=float(roc_auc_score(yp, pt_)), ks=ks_stat(yp, pt_),
                             **{k: v for k, v in fr.items() if not isinstance(v, (dict, list))}, group_rates=fr["group_rates"]))
    return grid
def pick(grid, rw, lam): return next(g for g in grid if g["reweigh"] == rw and abs(g["lam"]-lam) < 1e-9)
grid = run_grid(p_lr_va, P["logreg"], pv_lr_rw, P["logreg_rw"]); grid_x = run_grid(p_xb_va, P["xgb_tuned"], pv_rw, P["xgb_tuned_rw"])
res["fairness_grid"] = grid
res["fairness_summary"] = {k: pick(grid, rw, lam) for k, (rw, lam) in dict(none=(0, 0.0), half=(0, .5), calibrated=(0, 1.0), reweighed=(1, 0.0), both=(1, 1.0)).items()}
res["fairness_summary_xgb"] = {k: pick(grid_x, rw, lam) for k, (rw, lam) in dict(none=(0, 0.0), calibrated=(0, 1.0), reweighed=(1, 0.0), both=(1, 1.0)).items()}
res["fairness_seconds"] = time.time()-t0
log_run("fairness_logreg_reweigh_plus_threshold", "LogReg+Reweigh+GroupThr", dict(C=1.0, reweigh=True, calibration=1.0), m_lr_va, {"stage": "fairness"},
        test_m=metrics(Y[i_te], P["logreg_rw"], .3), dur=res["fairness_seconds"])

### 9. SHAP-style importance (XGB gain)

In [ ]:
res["importance"] = dict(zip(FEATS, map(float, xgb_best.feature_importances_)))

### 10. Plots

In [ ]:
NAVY, TEAL, ORANGE, GREY = "#12355B", "#0F8B8D", "#E07A1F", "#8A97A6"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False, "font.family": "DejaVu Sans"})
# (a) Optuna history
vals = np.array([t.value for t in study.trials]); best = np.maximum.accumulate(vals)
fig, ax = plt.subplots(figsize=(7, 3.8)); ax.scatter(range(len(vals)), vals, c=GREY, s=28, label="Trial AUC")
ax.plot(best, c=TEAL, lw=2.5, label="Best so far"); ax.set_xlabel("Trial"); ax.set_ylabel("Validation AUC"); ax.legend(frameon=False)
ax.set_ylim(min(vals)-.002, max(vals)+.002); ax.set_title("Optuna optimisation history (40 trials)", loc="left", color=NAVY, weight="bold")
fig.tight_layout(); fig.savefig(f"{OUT}/optuna_history.png", dpi=200); plt.show()
# (b) Optuna param importance
fig, ax = plt.subplots(figsize=(6, 3.8)); ks_ = list(imp)[::-1]; ax.barh(ks_, [imp[k] for k in ks_], color=TEAL)
ax.set_xlabel("Importance"); ax.set_title("Hyperparameter importance", loc="left", color=NAVY, weight="bold"); fig.tight_layout(); fig.savefig(f"{OUT}/optuna_importance.png", dpi=200); plt.show()
# (c) Reliability diagram
fig, axs = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, k, ttl in zip(axs, ["xgb_tuned", "xgb_tuned_cal"], ["Tuned XGBoost (raw)", "After isotonic calibration"]):
    fr_, mp_ = calibration_curve(Y[i_te], P[k], n_bins=10, strategy="quantile")
    ax.plot([0, 1], [0, 1], "--", c=GREY, label="Perfect"); ax.plot(mp_, fr_, "o-", c=TEAL if "cal" in k else ORANGE, lw=2.2, label="Model")
    ax.set_title(f"{ttl}\nBrier = {res['test'][k]['brier']:.4f}", color=NAVY, weight="bold", fontsize=11, loc="left"); ax.set_xlabel("Mean predicted default probability"); ax.legend(frameon=False)
axs[0].set_ylabel("Observed default rate"); fig.tight_layout(); fig.savefig(f"{OUT}/reliability.png", dpi=200); plt.show()
res["calibration_curve"] = {k: [list(map(float, a)) for a in calibration_curve(Y[i_te], P[k], n_bins=10, strategy="quantile")] for k in ["xgb_tuned", "xgb_tuned_cal"]}
# (d) ROC comparison
fig, ax = plt.subplots(figsize=(5.4, 4.4))
for k, c, lb in [("logreg", GREY, "Logistic regression (baseline)"), ("xgb_tuned", TEAL, "Tuned XGBoost")]:
    fpr, tpr, _ = roc_curve(Y[i_te], P[k]); ax.plot(fpr, tpr, c=c, lw=2.3, label=f"{lb}  AUC {res['test'][k]['auc']:.3f}")
ax.plot([0, 1], [0, 1], "--", c="#c4ccd6"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(frameon=False, loc="lower right", fontsize=9)
ax.set_title("ROC on held-out test set", loc="left", color=NAVY, weight="bold"); fig.tight_layout(); fig.savefig(f"{OUT}/roc.png", dpi=200); plt.show()
# (e) Pareto frontier
fig, ax = plt.subplots(figsize=(6.4, 4))
for rw, c, lb in [(0, GREY, "No reweighing"), (1, TEAL, "Reweighed")]:
    gg = [g for g in grid if g["reweigh"] == rw]; ax.plot([g["dir"] for g in gg], [g["ba"] for g in gg], "o-", c=c, label=lb, lw=2)
ax.axvline(.8, ls="--", c=NAVY); ax.text(.805, ax.get_ylim()[0]+.002, "DIR = 0.80", color=NAVY, fontsize=9)
ax.set_xlabel("Disparate impact ratio (fairer →)"); ax.set_ylabel("Balanced accuracy"); ax.legend(frameon=False)
ax.set_title("Accuracy-fairness Pareto frontier (test)", loc="left", color=NAVY, weight="bold"); fig.tight_layout(); fig.savefig(f"{OUT}/pareto.png", dpi=200); plt.show()
# (f) Confusion matrix (tuned XGB)
m = res["test"]["xgb_tuned"]; cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
fig, ax = plt.subplots(figsize=(4.2, 3.8)); ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=16, color="white" if cm[i, j] > cm.max()/2 else NAVY, weight="bold")
ax.set_xticks([0, 1], ["Pred: repay", "Pred: default"]); ax.set_yticks([0, 1], ["Actual: repay", "Actual: default"]); ax.spines[:].set_visible(False)
ax.set_title("Confusion matrix – tuned XGBoost (test)", color=NAVY, weight="bold", fontsize=10); fig.tight_layout(); fig.savefig(f"{OUT}/confusion.png", dpi=200); plt.show()
# (g) MLflow-style dashboard (runs table + metric comparison) rendered from the real MLflow store
runs = mlflow.search_runs(order_by=["metrics.val_auc DESC"]); runs.to_csv(f"{OUT}/mlflow_runs.csv", index=False)
res["mlflow"] = dict(n_runs=int(len(runs)), best_run=str(runs.iloc[0]["tags.mlflow.runName"]), best_val_auc=float(runs.iloc[0]["metrics.val_auc"]))

json.dump(res, open(f"{OUT}/results.json", "w"), indent=1, default=float)
print(json.dumps({k: res[k] for k in ("data", "split", "optuna", "test", "fairness_summary", "overfit", "mlflow", "test_train_seconds")}, indent=1, default=float)[:9000])

### Summary of Part 1

In [ ]:
import json
r = json.load(open('results/results.json'))
print('Selected model:', r['selected_model'])
print('Test AUC  logreg: %.3f | tuned XGBoost: %.3f' % (r['test']['logreg']['auc'], r['test']['xgb_tuned']['auc']))
rel = next(g for g in r['fairness_grid'] if g['reweigh']==1 and abs(g['lam']-0.6)<1e-9)
print('DIR no controls: %.2f | reweigh + 60%% thresholds: %.2f' % (r['fairness_summary']['none']['dir'], rel['dir']))

## Part 2. EDA, data quality, Fairlearn, SHAP, DiCE, drift and packaging

In [ ]:
"""Part 2: EDA, data quality, Fairlearn, group calibration, error analysis, SHAP, DiCE, drift, model packaging.
Re-creates the identical split (seed 42) and re-trains the selected logistic regression; asserts it reproduces test AUC from pipeline.py."""
import json, hashlib, joblib, warnings, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix
from fairlearn.metrics import (MetricFrame, selection_rate, demographic_parity_difference, demographic_parity_ratio,
                               equalized_odds_difference, true_positive_rate, false_positive_rate)
import shap
warnings.filterwarnings("ignore")
OUT = "results"; SEED = 42; R = json.load(open(f"{OUT}/results.json"))
NAVY, TEAL, ORANGE, GREY = "#12355B", "#0F8B8D", "#E07A1F", "#8A97A6"
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
df = pd.read_csv(f"{OUT}/dataset.csv"); FEATS = R["data"]["features"]; out = {}
out["dataset_md5"] = hashlib.md5(open(f"{OUT}/dataset.csv", "rb").read()).hexdigest()
LABEL = {"recharge_regularity": "Recharge regularity", "mobile_money_txn_log": "Mobile-money txns (log)", "utility_on_time": "Utility bills on time",
         "shg_repayment": "SHG repayment", "earlier_unpaid_loan": "Earlier unpaid loan", "loan_amount_k": "Loan amount (k INR)",
         "monthly_income_k": "Monthly income (k INR)", "informal_income": "Informal income", "basic_phone": "Basic phone"}


### split (identical to pipeline.py)

In [ ]:
N = len(df); strat = df["default"]*4 + df.female*2 + df.rural; idx = np.arange(N)
i_tr, i_tmp = train_test_split(idx, test_size=.4, stratify=strat, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=.5, stratify=strat.iloc[i_tmp], random_state=SEED)
X, Y = df[FEATS].values, df["default"].values; G = (df.female*2 + df.rural).values

### 1. Data quality

In [ ]:
dq = dict(n_rows=N, n_cols=df.shape[1], missing_total=int(df.isna().sum().sum()), duplicates=int(df.duplicated().sum()),
          dtypes={c: str(t) for c, t in df.dtypes.items()})
rng_checks = {"recharge_regularity in [0,1]": bool(df.recharge_regularity.between(0, 1).all()), "utility_on_time in [0,1]": bool(df.utility_on_time.between(0, 1).all()),
              "shg_repayment in [0,1]": bool(df.shg_repayment.between(0, 1).all()), "loan_amount_k in [3,60]": bool(df.loan_amount_k.between(3, 60).all()),
              "binary cols in {0,1}": bool(df[["earlier_unpaid_loan", "informal_income", "basic_phone", "default", "female", "rural"]].isin([0, 1]).all().all()),
              "monthly_income_k > 0": bool((df.monthly_income_k > 0).all())}
dq["range_checks"] = rng_checks; out["dq"] = dq
# descriptive stats
rows = []
for c in FEATS:
    s = df[c]; q1, q3 = s.quantile(.25), s.quantile(.75); iqr = q3-q1
    lo, hi = q1-1.5*iqr, q3+1.5*iqr; z = np.abs(stats.zscore(s)); 
    rows.append(dict(feature=c, mean=s.mean(), median=s.median(), variance=s.var(), std=s.std(), min=s.min(), max=s.max(), q1=q1, q3=q3, iqr=iqr,
                     skew=float(stats.skew(s)), iqr_outliers=int(((s < lo) | (s > hi)).sum()), z3_outliers=int((z > 3).sum())))
eda = pd.DataFrame(rows); eda.to_csv(f"{OUT}/eda_stats.csv", index=False); out["eda"] = eda.to_dict("records")
# missingness MCAR-style check: none missing -> report only
# plots: histograms
fig, axs = plt.subplots(3, 3, figsize=(8.5, 6.8))
for ax, c in zip(axs.ravel(), FEATS):
    if df[c].nunique() <= 2:
        vc = df[c].value_counts(normalize=True).sort_index(); ax.bar(["0", "1"], vc.values, color=TEAL)
    else: ax.hist(df[c], bins=40, color=TEAL, edgecolor="white", linewidth=.3)
    ax.set_title(LABEL[c], fontsize=9)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_hist.png", dpi=170); plt.show()
# boxplots by default class
cont = [c for c in FEATS if df[c].nunique() > 2]
fig, axs = plt.subplots(2, 3, figsize=(8.5, 4.8))
for ax, c in zip(axs.ravel(), cont):
    ax.boxplot([df[c][df.default == 0], df[c][df.default == 1]], tick_labels=["Repaid", "Default"], patch_artist=True, boxprops=dict(facecolor="#CFE8E8"), medianprops=dict(color=ORANGE), flierprops=dict(markersize=2, alpha=.4))
    ax.set_title(LABEL[c], fontsize=9)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_box.png", dpi=170); plt.show()
# correlation heatmap
cc = df[FEATS+["female", "rural", "default"]].corr(); lab = [LABEL.get(c, c.capitalize()) for c in cc.columns]
fig, ax = plt.subplots(figsize=(7.6, 6.4)); im = ax.imshow(cc, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(lab)), lab, rotation=60, ha="right", fontsize=8); ax.set_yticks(range(len(lab)), lab, fontsize=8)
for i in range(len(lab)):
    for j in range(len(lab)): ax.text(j, i, f"{cc.iloc[i,j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if abs(cc.iloc[i, j]) > .5 else "black")
fig.colorbar(im, shrink=.8); fig.tight_layout(); fig.savefig(f"{OUT}/fig_corr.png", dpi=170); plt.show()
out["corr_target"] = cc["default"].drop("default").to_dict(); out["corr_female"] = cc["female"].drop("female").to_dict(); out["corr_rural"] = cc["rural"].drop("rural").to_dict()
# VIF
from numpy.linalg import inv
Z = StandardScaler().fit_transform(df[FEATS]); vif = np.diag(inv(np.corrcoef(Z.T))); out["vif"] = dict(zip(FEATS, map(float, vif)))
# group default-rate table + chi-square tests
gt = {}
for nm, col in (("gender", "female"), ("region", "rural")):
    ct = pd.crosstab(df[col], df.default); chi2, p, _, _ = stats.chi2_contingency(ct); gt[nm] = dict(rate0=float(df.default[df[col] == 0].mean()), rate1=float(df.default[df[col] == 1].mean()), chi2=float(chi2), p=float(p))
out["group_tests"] = gt

### 2. Model packaging (reproduce selected LR)

In [ ]:
sc = StandardScaler().fit(X[i_tr]); Xs = sc.transform(X); lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xs[i_tr], Y[i_tr])
p_te = lr.predict_proba(Xs[i_te])[:, 1]; p_va = lr.predict_proba(Xs[i_va])[:, 1]
auc_te = roc_auc_score(Y[i_te], p_te); assert abs(auc_te-R["test"]["logreg"]["auc"]) < 1e-9, (auc_te, R["test"]["logreg"]["auc"])
# release setting: reweighing + 60% threshold calibration (learned on validation)
def reweigh(ix):
    g = G[ix]; yy = Y[ix]; n = len(ix); pg = np.bincount(g, minlength=4)/n; py = np.bincount(yy, minlength=2)/n
    pgy = np.array([[((g == a) & (yy == b)).sum()/n for b in (0, 1)] for a in range(4)]); return np.array([pg[a]*py[b]/pgy[a, b] for a, b in zip(g, yy)])
lr_rw = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xs[i_tr], Y[i_tr], sample_weight=reweigh(i_tr))
pv_rw = lr_rw.predict_proba(Xs[i_va])[:, 1]; pt_rw = lr_rw.predict_proba(Xs[i_te])[:, 1]
APPROVE, LAM = .70, .60
def group_thr(pv, gv, lam):
    t0 = np.quantile(pv, APPROVE); return [float(np.quantile(pv[gv == g], min(.999, (1-lam)*(pv[gv == g] < t0).mean() + lam*APPROVE))) for g in range(4)]
thr_rel = group_thr(pv_rw, G[i_va], LAM); thr_none = group_thr(p_va, G[i_va], 0.0)
ap_rel = pt_rw < np.array(thr_rel)[G[i_te]]; ap_none = p_te < np.array(thr_none)[G[i_te]]
joblib.dump(dict(scaler=sc, model=lr_rw, features=FEATS, group_thresholds=thr_rel, borderline_band=.05, version="1.0.0", dataset_md5=out["dataset_md5"]), "deploy/model.joblib")
out["packaging"] = dict(thr_release=thr_rel, thr_no_control=thr_none, coefs=dict(zip(FEATS, map(float, lr_rw.coef_[0]))), intercept=float(lr_rw.intercept_[0]),
                        coefs_unweighted=dict(zip(FEATS, map(float, lr.coef_[0]))), size_kb=os.path.getsize("deploy/model.joblib")/1024)

### 3. Fairlearn

In [ ]:
fl = {}
for tag, ap in (("no_control", ap_none), ("release", ap_rel)):
    d = {}
    for nm, col in (("gender", df.female.values[i_te]), ("region", df.rural.values[i_te])):
        mf = MetricFrame(metrics=dict(selection_rate=selection_rate, tpr=true_positive_rate, fpr=false_positive_rate), y_true=1-Y[i_te], y_pred=ap.astype(int), sensitive_features=col)
        d[nm] = dict(by_group=mf.by_group.to_dict("index"), dpd=float(demographic_parity_difference(1-Y[i_te], ap.astype(int), sensitive_features=col)),
                     dpr=float(demographic_parity_ratio(1-Y[i_te], ap.astype(int), sensitive_features=col)), eod=float(equalized_odds_difference(1-Y[i_te], ap.astype(int), sensitive_features=col)))
    fl[tag] = d
out["fairlearn"] = fl

### 4. Calibration across groups (selected LR, raw)

In [ ]:
names = ["Male-Urban", "Male-Rural", "Female-Urban", "Female-Rural"]; cg = []
for g in range(4):
    m = G[i_te] == g; cg.append(dict(group=names[g], n=int(m.sum()), observed=float(Y[i_te][m].mean()), predicted=float(p_te[m].mean()), brier=float(brier_score_loss(Y[i_te][m], p_te[m])), auc=float(roc_auc_score(Y[i_te][m], p_te[m]))))
out["group_calibration"] = cg
fig, ax = plt.subplots(figsize=(5.4, 3.6)); xx = np.arange(4); ax.bar(xx-.2, [c["observed"] for c in cg], .4, color=NAVY, label="Observed default rate"); ax.bar(xx+.2, [c["predicted"] for c in cg], .4, color=TEAL, label="Mean predicted risk")
ax.set_xticks(xx, names, fontsize=8.5); ax.set_ylabel("Default rate / risk"); ax.legend(frameon=False, fontsize=8.5); fig.tight_layout(); fig.savefig(f"{OUT}/fig_groupcal.png", dpi=170); plt.show()

### 5. Error analysis (LR, threshold 0.26)

In [ ]:
thr = R["test"]["logreg"]["thr"]; pred = (p_te >= thr).astype(int); yt = Y[i_te]; Xt = X[i_te]
ea = {}
for g in range(4):
    m = G[i_te] == g; tn, fp, fn, tp = confusion_matrix(yt[m], pred[m]).ravel(); ea[names[g]] = dict(n=int(m.sum()), fnr=float(fn/(fn+tp)), fpr=float(fp/(fp+tn)), precision=float(tp/(tp+fp)), fn=int(fn), fp=int(fp))
out["error_by_group"] = ea
fnm = (yt == 1) & (pred == 0); tpm = (yt == 1) & (pred == 1); fpm = (yt == 0) & (pred == 1); tnm = (yt == 0) & (pred == 0)
out["error_feature_means"] = {c: dict(FN=float(Xt[fnm, j].mean()), TP=float(Xt[tpm, j].mean()), FP=float(Xt[fpm, j].mean()), TN=float(Xt[tnm, j].mean())) for j, c in enumerate(FEATS)}
out["error_counts"] = dict(FN=int(fnm.sum()), TP=int(tpm.sum()), FP=int(fpm.sum()), TN=int(tnm.sum()), borderline_share=float((np.abs(p_te-thr) < .05).mean()), fn_borderline=float((np.abs(p_te[fnm]-thr) < .1).mean()))

### 6. SHAP

In [ ]:
bg = Xs[i_tr][:500]; expl = shap.LinearExplainer(lr, bg); sv = expl.shap_values(Xs[i_te][:2000])
plt.figure(figsize=(7, 4.2)); shap.summary_plot(sv, Xs[i_te][:2000], feature_names=[LABEL[c] for c in FEATS], show=False, plot_size=None); plt.tight_layout(); plt.savefig(f"{OUT}/fig_shap_beeswarm.png", dpi=170); plt.show()
ma = np.abs(sv).mean(0); o = np.argsort(ma)
fig, ax = plt.subplots(figsize=(6.4, 3.6)); ax.barh([LABEL[FEATS[j]] for j in o], ma[o], color=TEAL); ax.set_xlabel("Mean |SHAP value| (log-odds)"); fig.tight_layout(); fig.savefig(f"{OUT}/fig_shap_bar.png", dpi=170); plt.show()
out["shap_mean_abs"] = dict(zip(FEATS, map(float, ma)))
# verification: SHAP == coef*(x-mean)/sd for linear model
ex = lr.coef_[0]*(Xs[i_te][:2000]-bg.mean(0)); out["shap_linear_check_maxdiff"] = float(np.abs(ex-sv).max()); out["shap_check_info"] = dict(sv_shape=list(np.shape(sv)), ex_shape=list(ex.shape), corr=float(np.corrcoef(ex.ravel(), np.asarray(sv).ravel())[0,1]))

### 7. DiCE counterfactual

In [ ]:
try:
    import dice_ml
    from sklearn.pipeline import Pipeline
    pipe = Pipeline([("sc", sc), ("lr", lr)]); dtr = pd.DataFrame(X[i_tr], columns=FEATS); dtr["default"] = Y[i_tr]
    dd = dice_ml.Data(dataframe=dtr, continuous_features=["recharge_regularity", "mobile_money_txn_log", "utility_on_time", "shg_repayment", "loan_amount_k", "monthly_income_k"], outcome_name="default")
    CUT = .20; sh = np.log(CUT/(1-CUT))
    def adj(d):  # shift logit so that class flips at the illustrative 20% risk cut-off
        p = pipe.predict_proba(d[FEATS].values)[:, 1]; z = np.log(p/(1-p))-sh; q = 1/(1+np.exp(-z)); return np.c_[1-q, q]
    class W:
        def predict_proba(self, d): return adj(d)
        def predict(self, d): return (adj(d)[:, 1] > .5).astype(int)
    dm = dice_ml.Model(model=W(), backend="sklearn"); ex_ = dice_ml.Dice(dd, dm, method="random")
    pt = pipe.predict_proba(X[i_te])[:, 1]; cand = np.where((pt > .22) & (pt < .30))[0][3]
    q = pd.DataFrame([X[i_te][cand]], columns=FEATS)
    cf = ex_.generate_counterfactuals(q, total_CFs=3, desired_class=0, features_to_vary=["recharge_regularity", "utility_on_time", "shg_repayment", "loan_amount_k"],
                                      permitted_range={"recharge_regularity": [0, 1], "utility_on_time": [0, 1], "shg_repayment": [0, 1], "loan_amount_k": [3, 60]})
    cfd = cf.cf_examples_list[0].final_cfs_df; cfd = cfd[FEATS]
    out["dice"] = dict(cutoff=CUT, original=q.iloc[0].to_dict(), original_p=float(pipe.predict_proba(q.values)[0, 1]), cfs=cfd.to_dict("records"), cf_p=[float(v) for v in pipe.predict_proba(cfd.values)[:, 1]])
except Exception as e:
    out["dice"] = dict(error=repr(e))

### 8. Drift (simulated)

In [ ]:
def psi(a, b, bins=10):
    e = np.unique(np.quantile(a, np.linspace(0, 1, bins+1))); e[0], e[-1] = -np.inf, np.inf
    pa = np.histogram(a, e)[0]/len(a); pb = np.histogram(b, e)[0]/len(b); pa, pb = np.clip(pa, 1e-4, None), np.clip(pb, 1e-4, None); return float(np.sum((pb-pa)*np.log(pb/pa)))
base = X[i_tr]; same = X[i_te].copy(); drift = X[i_te].copy()
j_rc = FEATS.index("recharge_regularity"); j_mm = FEATS.index("mobile_money_txn_log")
drift[:, j_rc] = np.clip(drift[:, j_rc]-.10, 0, 1); drift[:, j_mm] = drift[:, j_mm]-.4   # simulated seasonal shock
out["psi_stable"] = {c: psi(base[:, j], same[:, j]) for j, c in enumerate(FEATS)}; out["psi_drift"] = {c: psi(base[:, j], drift[:, j]) for j, c in enumerate(FEATS)}
pd_s = lr.predict_proba(sc.transform(drift))[:, 1]; out["drift_effect"] = dict(auc_stable=float(auc_te), auc_drift=float(roc_auc_score(Y[i_te], pd_s)), mean_risk_stable=float(p_te.mean()), mean_risk_drift=float(pd_s.mean()))
fig, ax = plt.subplots(figsize=(7, 3.6)); xx = np.arange(len(FEATS))
ax.bar(xx-.2, [out["psi_stable"][c] for c in FEATS], .4, color=GREY, label="Held-out test (stable)"); ax.bar(xx+.2, [out["psi_drift"][c] for c in FEATS], .4, color=ORANGE, label="Simulated seasonal shift")
ax.axhline(.1, ls="--", c=NAVY, lw=1); ax.axhline(.25, ls="--", c="#C0392B", lw=1); ax.text(8.6, .105, "0.10", fontsize=8); ax.text(8.6, .255, "0.25", fontsize=8)
ax.set_xticks(xx, [LABEL[c] for c in FEATS], rotation=40, ha="right", fontsize=8); ax.set_ylabel("PSI"); ax.legend(frameon=False, fontsize=8.5); fig.tight_layout(); fig.savefig(f"{OUT}/fig_psi.png", dpi=170); plt.show()

### 9. Preprocessing before/after

In [ ]:
j = FEATS.index("mobile_money_txn_log"); raw_mm = np.expm1(df.mobile_money_txn_log)
out["transform_example"] = dict(raw_skew=float(stats.skew(raw_mm)), log_skew=float(stats.skew(df.mobile_money_txn_log)), raw_mean=float(raw_mm.mean()), raw_std=float(raw_mm.std()),
                                example_raw=float(raw_mm.iloc[0]), example_log=float(df.mobile_money_txn_log.iloc[0]), mu_train=float(sc.mean_[j]), sd_train=float(sc.scale_[j]))
fig, axs = plt.subplots(1, 3, figsize=(8.5, 2.8))
axs[0].hist(raw_mm, bins=40, color=GREY); axs[0].set_title("Raw count (skew %.2f)" % out["transform_example"]["raw_skew"], fontsize=9)
axs[1].hist(df.mobile_money_txn_log, bins=40, color=TEAL); axs[1].set_title("log(1+x) (skew %.2f)" % out["transform_example"]["log_skew"], fontsize=9)
axs[2].hist(Xs[i_tr][:, j], bins=40, color=NAVY); axs[2].set_title("Standardised (train)", fontsize=9); fig.tight_layout(); fig.savefig(f"{OUT}/fig_transform.png", dpi=170); plt.show()
# worked example row
r0 = df.iloc[i_te[0]]; out["worked_example"] = dict(raw={c: float(r0[c]) for c in FEATS}, z={c: float(Xs[i_te[0], k]) for k, c in enumerate(FEATS)}, p=float(p_te[0]),
                                                  contrib={c: float(lr.coef_[0][k]*Xs[i_te[0], k]) for k, c in enumerate(FEATS)}, intercept=float(lr.intercept_[0]))
json.dump(out, open(f"{OUT}/results2.json", "w"), indent=1, default=float)
print(json.dumps({k: out[k] for k in ("dq", "group_tests", "vif", "fairlearn", "group_calibration", "error_by_group", "error_counts", "drift_effect", "dice", "shap_linear_check_maxdiff", "packaging")}, indent=1, default=float)[:7000])

## Part 3. Try the trained model
Change the applicant and re-run. This mirrors the FastAPI `/score` endpoint and the HTML demo.

In [ ]:
import joblib, numpy as np
A = joblib.load('deploy/model.joblib')
def score(recharge_regularity=.42, mobile_money_txn=5, utility_on_time=.68, shg_repayment=.82, earlier_unpaid_loan=0,
          loan_amount_k=12, monthly_income_k=9, informal_income=1, basic_phone=1, female=1, rural=1):
    x = np.array([[recharge_regularity, np.log1p(mobile_money_txn), utility_on_time, shg_repayment, earlier_unpaid_loan, loan_amount_k, monthly_income_k, informal_income, basic_phone]])
    z = A['scaler'].transform(x); p = float(A['model'].predict_proba(z)[0, 1]); t = A['group_thresholds'][female*2 + rural]
    top = [A['features'][j] for j in np.argsort(-(A['model'].coef_[0] * z[0]))[:3]]
    return dict(default_probability=round(p, 4), threshold=round(t, 4), decision='APPROVE' if p < t else 'DECLINE', manual_review=abs(p-t) <= .05, top_reasons=top)
print(score())
print(score(recharge_regularity=.25, mobile_money_txn=2, utility_on_time=.4, shg_repayment=.35, earlier_unpaid_loan=1, loan_amount_k=48, monthly_income_k=7, female=0, rural=1))

In [ ]:
import mlflow
mlflow.set_tracking_uri('sqlite:///' + os.path.abspath('mlflow.db'))
runs = mlflow.search_runs(experiment_names=['FairLend-credit-default'], order_by=['metrics.val_auc DESC'])
print(len(runs), 'runs tracked')
print(runs[['tags.mlflow.runName', 'metrics.val_auc', 'metrics.val_f1']].head(10).to_string())
!zip -rq fairlend_colab_results.zip results deploy mlflow.db
try:
    from google.colab import files; files.download('fairlend_colab_results.zip')
except ImportError:
    print('Not in Colab: results saved in fairlend_colab_results.zip')